# 02 · Data Preparation: one clean, validated warehouse

**Goal:** turn the 70 raw trip files into one consistent, traceable dataset, using the issue register from notebook 01.

| Step | What happens |
|---|---|
| 1 | Map every raw header (summer and winter) to one clean `snake_case` schema |
| 2 | Load all trips into one table, values unchanged (**raw layer**) |
| 3 | Apply the fixes from `reports/01_quality_issues.csv`, each one logged (**clean layer**) |
| 4 | Label each sample as `driving`, `stopped` or `charging` |
| 5 | Build a trip table from `Overview.xlsx` |
| 6 | Load everything into DuckDB and run data-quality checks |

Nothing beyond preparation happens here: no feature engineering, no EDA, no models.

**Output:** `data/warehouse/volvo.duckdb` with schemas `raw`, `clean` and `meta`, plus `reports/02_cleaning_log.csv`
and `reports/02_quality_checks.csv`.

In [1]:
from pathlib import Path
import re

import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
WAREHOUSE = ROOT / "data" / "warehouse"
REPORTS = ROOT / "reports"
WAREHOUSE.mkdir(parents=True, exist_ok=True)
DB_PATH = WAREHOUSE / "volvo.duckdb"
HZ = 10  # samples per second, confirmed in notebook 01

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

register = pd.read_csv(REPORTS / "01_quality_issues.csv")
print(f"issues carried over from notebook 01: {len(register)}")
register[["issue", "scope", "severity"]]

issues carried over from notebook 01: 22


,issue,scope,severity
0,Missing values in key signals,TripB34,high
1,Missing values in key signals,TripB28,high
2,Missing values in key signals,TripB09,high
3,Missing values in key signals,TripB07,high
4,Value outside physical limits,TripB14,medium
5,Missing values in key signals,TripB13,medium
6,Heating power spike,TripB02,medium
7,SoC jump between consecutive samples,TripB14,medium
8,Trips contain charging phases,"TripB04, TripB09",medium
9,Header: malformed unit bracket,"70 trips (e.g. TripA01, TripA02, TripA03)",low


## 1 · One schema for summer and winter trips

Every raw header is mapped by hand to a clean name with its unit. The map handles the problems found in notebook 01:
- `Velocity [km/h]]]` (TripB38) and `Velocity [km/h]` map to the same column
- `max. SoC [%)` gets a proper name
- the empty trailing header in TripA11 is dropped
- `Temperature Vent right [°C]` appears **twice** in winter trips. Columns are mapped by position, so the second one
  becomes `vent_right_2_temp_c` instead of overwriting the first

In [2]:
COLUMN_MAP = {
    "Time [s]": "time_s",
    "Velocity [km/h]": "speed_kmh",
    "Velocity [km/h]]]": "speed_kmh",
    "Elevation [m]": "elevation_m",
    "Throttle [%]": "throttle_pct",
    "Motor Torque [Nm]": "motor_torque_nm",
    "Longitudinal Acceleration [m/s^2]": "accel_long_ms2",
    "Regenerative Braking Signal": "regen_signal",
    "Battery Voltage [V]": "batt_voltage_v",
    "Battery Current [A]": "batt_current_a",
    "Battery Temperature [°C]": "batt_temp_c",
    "max. Battery Temperature [°C]": "batt_temp_max_c",
    "SoC [%]": "soc_pct",
    "displayed SoC [%]": "soc_displayed_pct",
    "min. SoC [%]": "soc_min_pct",
    "max. SoC [%)": "soc_max_pct",
    "Heating Power CAN [kW]": "heating_power_can_kw",
    "Heating Power LIN [W]": "heating_power_lin_w",
    "Requested Heating Power [W]": "heating_power_requested_w",
    "AirCon Power [kW]": "aircon_power_kw",
    "Heater Signal": "heater_signal",
    "Heater Voltage [V]": "heater_voltage_v",
    "Heater Current [A]": "heater_current_a",
    "Ambient Temperature [°C]": "ambient_temp_c",
    "Ambient Temperature Sensor [°C]": "ambient_temp_sensor_c",
    "Coolant Temperature Heatercore [°C]": "coolant_temp_heatercore_c",
    "Requested Coolant Temperature [°C]": "coolant_temp_requested_c",
    "Coolant Temperature Inlet [°C]": "coolant_temp_inlet_c",
    "Coolant Volume Flow +500 [l/h]": "coolant_flow_raw_lph",
    "Heat Exchanger Temperature [°C]": "heat_exchanger_temp_c",
    "Cabin Temperature Sensor [°C]": "cabin_temp_c",
    "Temperature Coolant Heater Inlet [°C]": "coolant_heater_inlet_temp_c",
    "Temperature Coolant Heater Outlet [°C]": "coolant_heater_outlet_temp_c",
    "Temperature Heat Exchanger Outlet [°C]": "heat_exchanger_outlet_temp_c",
    "Temperature Defrost lateral left [°C]": "defrost_lateral_left_temp_c",
    "Temperature Defrost lateral right [°C]": "defrost_lateral_right_temp_c",
    "Temperature Defrost central [°C]": "defrost_central_temp_c",
    "Temperature Defrost central left [°C]": "defrost_central_left_temp_c",
    "Temperature Defrost central right [°C]": "defrost_central_right_temp_c",
    "Temperature Footweel Driver [°C]": "footwell_driver_temp_c",
    "Temperature Footweel Co-Driver [°C]": "footwell_codriver_temp_c",
    "Temperature Feetvent Co-Driver [°C]": "feetvent_codriver_temp_c",
    "Temperature Feetvent Driver [°C]": "feetvent_driver_temp_c",
    "Temperature Head Co-Driver [°C]": "head_codriver_temp_c",
    "Temperature Head Driver [°C]": "head_driver_temp_c",
    "Temperature Vent right [°C]": "vent_right_temp_c",
    "Temperature Vent central right [°C]": "vent_central_right_temp_c",
    "Temperature Vent central left [°C]": "vent_central_left_temp_c",
    "": None,  # empty trailing header (TripA11): dropped
}


def raw_header(path):
    with open(path, encoding="latin-1") as f:
        return [c.strip() for c in f.readline().rstrip("\r\n").split(";")]


def clean_names(header):
    names, seen = [], {}
    for h in header:
        name = COLUMN_MAP[h]  # KeyError here means an unmapped header
        if name is not None:
            seen[name] = seen.get(name, 0) + 1
            name = name if seen[name] == 1 else name.replace("_temp_c", f"_{seen[name]}_temp_c")
        names.append(name)
    return names


trip_files = sorted(RAW.glob("Trip*.csv"))
headers = {p.stem: raw_header(p) for p in trip_files}
unmapped = sorted({h for hs in headers.values() for h in hs} - set(COLUMN_MAP))
assert not unmapped, f"unmapped headers: {unmapped}"
print(f"all {len({h for hs in headers.values() for h in hs})} distinct raw headers are mapped")

column_map = pd.DataFrame(
    [{"raw_header": h, "clean_name": n} for h, n in COLUMN_MAP.items()]
).fillna({"clean_name": "(dropped)"})
column_map

all 49 distinct raw headers are mapped


,raw_header,clean_name
0,Time [s],time_s
1,Velocity [km/h],speed_kmh
2,Velocity [km/h]]],speed_kmh
3,Elevation [m],elevation_m
4,Throttle [%],throttle_pct
5,Motor Torque [Nm],motor_torque_nm
6,Longitudinal Acceleration [m/s^2],accel_long_ms2
7,Regenerative Braking Signal,regen_signal
8,Battery Voltage [V],batt_voltage_v
9,Battery Current [A],batt_current_a


## 2 · Raw layer: all trips in one table, values unchanged

In [3]:
SIGNALS = list(dict.fromkeys(n for p in trip_files for n in clean_names(headers[p.stem]) if n))


def load_trip(path):
    names = clean_names(headers[path.stem])
    df = pd.read_csv(path, sep=";", encoding="latin-1", header=0, names=[n or "_drop" for n in names])
    df = df.drop(columns=[c for c in df.columns if c == "_drop"])
    df.insert(0, "trip_id", path.stem)
    df.insert(1, "series", path.stem[4])
    df.insert(2, "row_idx", np.arange(len(df)))
    return df.reindex(columns=["trip_id", "series", "row_idx"] + SIGNALS)


raw = pd.concat([load_trip(p) for p in trip_files], ignore_index=True)
print(f"raw layer: {len(raw):,} rows x {raw.shape[1]} columns, {raw.trip_id.nunique()} trips")

coverage = raw.groupby("series")[SIGNALS].apply(lambda g: g.notna().any()).T
winter_only = coverage[~coverage["A"] & coverage["B"]].index.tolist()
print(f"signals recorded only in winter trips ({len(winter_only)}): they stay empty (NULL) for summer trips")

raw layer: 1,094,793 rows x 51 columns, 70 trips


signals recorded only in winter trips (20): they stay empty (NULL) for summer trips


## 3 · Clean layer: fixes from the issue register

Every change is logged with the issue it addresses and the number of values it touched.

In [4]:
clean = raw.copy()
log = []


def record(issue, action, trips, values_changed):
    log.append(dict(issue=issue, action=action, trips=trips, values_changed=int(values_changed)))

### 3.1 Values outside physical limits → missing

The same broad limits as notebook 01. The only hit there was the final row of TripB14, where SoC drops to 0 for one sample.

In [5]:
LIMITS = {
    "batt_voltage_v": (250, 420),
    "batt_current_a": (-500, 250),
    "batt_temp_c": (-30, 60),
    "soc_pct": (1, 100),
    "speed_kmh": (0, 160),
    "ambient_temp_c": (-30, 45),
}
for col, (lo, hi) in LIMITS.items():
    bad = clean[col].notna() & ~clean[col].between(lo, hi)
    if bad.any():
        trips = ", ".join(clean.loc[bad, "trip_id"].unique())
        print(f"{col}: {bad.sum()} values outside [{lo}, {hi}] in {trips} -> set to missing")
        clean.loc[bad, col] = np.nan
        record("Value outside physical limits", f"{col} outside [{lo}, {hi}] set to missing", trips, bad.sum())

soc_pct: 1 values outside [1, 100] in TripB14 -> set to missing


### 3.2 SoC gaps: fill only what can be filled honestly

Each SoC gap is one block of consecutive missing samples. SoC changes slowly, so short gaps of up to 5 s are filled:
- **inside a trip:** linear interpolation between the values on either side
- **at the first or last sample of a trip:** the nearest recorded value (many trips miss SoC only in their very last sample)

Longer gaps are left missing and flagged, because a guessed SoC would contaminate the models.

In [6]:
MAX_FILL_S = 5  # longest gap that is interpolated


def gap_runs(mask):
    # (start, length) of each run of True values
    edges = np.diff(np.concatenate([[0], mask.astype(int), [0]]))
    starts, ends = np.where(edges == 1)[0], np.where(edges == -1)[0]
    return list(zip(starts, ends - starts))


gap_table = []
clean["soc_filled"] = False
clean["soc_gap"] = False
for t, g in clean.groupby("trip_id", sort=False):
    soc = g["soc_pct"]
    for start, length in gap_runs(soc.isna().to_numpy()):
        idx = g.index[start:start + length]  # only this gap's samples are written, never the trip's other gaps
        seconds = length / HZ
        at_edge = start == 0 or start + length == len(g)
        if seconds > MAX_FILL_S:
            action = "left missing, flagged"
            clean.loc[idx, "soc_gap"] = True
        elif at_edge:
            action = "nearest value (trip edge)"
            clean.loc[idx, "soc_pct"] = soc.ffill().bfill().loc[idx]
            clean.loc[idx, "soc_filled"] = True
        else:
            action = "interpolated"
            clean.loc[idx, "soc_pct"] = soc.interpolate(limit_area="inside").loc[idx]
            clean.loc[idx, "soc_filled"] = True
        gap_table.append(dict(trip_id=t, position="start" if start == 0 else "end" if start + length == len(g) else "inside",
                              start_min=round(start / HZ / 60, 1), length_s=round(seconds, 1), action=action))

gaps = pd.DataFrame(gap_table)
print(gaps.groupby("action").agg(gaps=("trip_id", "size"), seconds=("length_s", "sum")).to_string())
print("\ngaps longer than 1 s:")
print(gaps[gaps.length_s > 1].to_string(index=False))
for action, part in gaps.groupby("action"):
    samples = int(round(part.length_s.sum() * HZ))
    record("Missing values in key signals", f"SoC: {action} (gaps <= {MAX_FILL_S} s filled, longer flagged soc_gap)",
           ", ".join(part.trip_id.unique()), samples)

                           gaps  seconds
action                                  
left missing, flagged         4   3074.8
nearest value (trip edge)    25      4.6

gaps longer than 1 s:
trip_id position  start_min  length_s                    action
TripB07      end       31.2     417.5     left missing, flagged
TripB09      end       59.3    2054.1     left missing, flagged
TripB13      end        5.9       1.3 nearest value (trip edge)
TripB28      end       20.1     162.5     left missing, flagged
TripB34      end        4.9     440.7     left missing, flagged


### 3.3 Heating-power "spike" in TripB02: checked, kept

Notebook 01 flagged heating power up to about 39 kW in TripB02, against about 7 kW in every other trip.
Before removing it, check it against the independent signals that measure the same thing.

In [7]:
b02 = clean[clean.trip_id == "TripB02"]
high = b02.heating_power_lin_w > 14_000
compare = pd.DataFrame({
    "heating LIN (W)": b02.heating_power_lin_w,
    "heating CAN (W)": b02.heating_power_can_kw * 1000,
    "requested (W)": b02.heating_power_requested_w,
    "heater V x I (W)": b02.heater_voltage_v * b02.heater_current_a,
})
print(f"samples above 14 kW: {high.sum()} (between {b02.time_s[high].min():.0f} s and {b02.time_s[high].max():.0f} s)")
print("\nmedian during those samples:")
print(compare[high].median().round(0).to_string())
print("\ncorrelation of LIN power with heater V x I over the whole trip:",
      round(compare["heating LIN (W)"].corr(compare["heater V x I (W)"]), 3))

samples above 14 kW: 1273 (between 72 s and 199 s)

median during those samples:
heating LIN (W)     26440.0
heating CAN (W)     27160.0
requested (W)       26234.0
heater V x I (W)    25298.0

correlation of LIN power with heater V x I over the whole trip: 1.0


In [8]:
clean["heating_extreme"] = clean.heating_power_lin_w > 14_000
record("Heating power spike", "kept: four independent signals agree, so it is a real event; "
       "flagged in heating_extreme for the anomaly step", "TripB02", 0)

### 3.4 Coolant flow offset

The header reads `Coolant Volume Flow +500 [l/h]`: the logged value is the flow minus 500 l/h (values sit between
about −500 and 0). The clean layer adds `coolant_flow_lph` = raw + 500 and keeps the raw column.
Two trips (B05, B06) never logged this signal; they stay missing.

In [9]:
clean["coolant_flow_lph"] = clean.coolant_flow_raw_lph + 500
print(clean.groupby("series").coolant_flow_lph.describe()[["count", "min", "50%", "max"]].round(1).to_string())
record("Column entirely empty", "coolant flow never logged in these trips: left missing", "TripB05, TripB06", 0)
record("Coolant flow offset", "added coolant_flow_lph = raw + 500 (raw column kept)", "winter trips",
       clean.coolant_flow_lph.notna().sum())

           count  min    50%    max
series                             
A            0.0  NaN    NaN    NaN
B       598759.0 -0.9  499.8  500.2


### 3.5 Stale CAN heating-power readings (found during EDA)

The EDA in notebook 04 showed a summer trip (TripA06, about 30 °C outside) with a mean "heating power" of 5.3 kW.
In that trip the CAN heating signal stays frozen at 5.32 kW while the **requested** heating power is 0 and the battery
current is far too low to feed a 5 kW heater. The CAN value was held, not measured.

**Rule:** CAN heating power above 0.5 kW while the requested heating power is below 100 W is set to missing and
flagged `heating_can_invalid`. The check below shows the rule only fires in summer trips; in winter, the CAN value
agrees with the request and the LIN signal.

In [10]:
stale = (clean.heating_power_can_kw > 0.5) & (clean.heating_power_requested_w < 100)
print(clean.assign(stale=stale).groupby("series").stale.mean().map("{:.2%}".format).to_string())
by_trip = clean.assign(stale=stale).groupby("trip_id").stale.mean()
print("\ntrips where more than 5 % of samples are affected:")
print(by_trip[by_trip > 0.05].map("{:.0%}".format).to_string())
clean["heating_can_invalid"] = stale
clean.loc[stale, "heating_power_can_kw"] = np.nan
record("Stale CAN heating power (found in notebook 04)",
       "heating_power_can_kw > 0.5 kW while requested < 100 W set to missing, flagged heating_can_invalid",
       ", ".join(by_trip[by_trip > 0.05].index), stale.sum())

series
A    16.54%
B     0.01%



trips where more than 5 % of samples are affected:
trip_id
TripA03     82%
TripA04      7%
TripA05     77%
TripA06    100%
TripA08      7%
TripA09      5%
TripA13     79%
TripA14      5%
TripA18      6%
TripA20     85%


## 4 · Phase labels: driving, stopped, charging

| Phase | Rule |
|---|---|
| `driving` | speed above 1 km/h |
| `charging` | speed at most 1 km/h, current above +20 A held for at least 60 s (regeneration needs motion, so this is plug-in charging) |
| `stopped` | everything else: traffic lights, parking, standing with the heater on |

In [11]:
MIN_CHARGE_S = 60


def label_phases(g):
    moving = g.speed_kmh.fillna(0) > 1
    charging = (~moving & (g.batt_current_a > 20)).to_numpy()
    keep = np.zeros(len(g), dtype=bool)
    for start, length in gap_runs(charging):
        if length >= MIN_CHARGE_S * HZ:
            keep[start:start + length] = True
    return pd.Series(np.where(moving, "driving", np.where(keep, "charging", "stopped")), index=g.index)


clean["phase"] = pd.concat([label_phases(g) for _, g in clean.groupby("trip_id", sort=False)])
phase_minutes = (clean.groupby(["trip_id", "phase"]).size() / HZ / 60).unstack(fill_value=0).round(1)
print("total hours per phase:")
print((phase_minutes.sum() / 60).round(1).to_string())
print("\ntrips with charging:")
print(phase_minutes[phase_minutes.get("charging", 0) > 0].to_string())
record("Trips contain charging phases", f"phase label added (charging = parked, > +20 A for >= {MIN_CHARGE_S} s)",
       ", ".join(phase_minutes[phase_minutes.get("charging", 0) > 0].index), (clean.phase == "charging").sum())

total hours per phase:
phase
charging     1.1
driving     25.6
stopped      3.7

trips with charging:
phase    charging  driving  stopped
trip_id                            
TripB04      23.5     20.3      5.4
TripB09      42.1     44.3      7.1


## 5 · Trip table from `Overview.xlsx`

In [12]:
ov = pd.read_excel(RAW / "Overview.xlsx", header=0).dropna(how="all")
ov = ov.drop(columns=[c for c in ov.columns if str(c).startswith("Unnamed") and ov[c].isna().all()])
ov.columns = [str(c).strip() for c in ov.columns]
rename = {
    "Trip": "trip_id", "Date": "start_raw", "Route/Area": "route", "Weather": "weather",
    "Battery Temperature (Start) [°C]": "batt_temp_start_c", "Battery Temperature (End)": "batt_temp_end_c",
    "Battery State of Charge (Start)": "soc_start_frac", "Battery State of Charge (End)": "soc_end_frac",
    "Unnamed: 8": "soc_used_frac", "Ambient Temperature (Start) [°C]": "ambient_temp_start_c",
    "Target Cabin Temperature": "target_cabin_temp_c", "Distance [km]": "distance_km",
    "Duration [min]": "duration_min_overview", "Fan": "fan", "Note": "note",
}
missing_names = [c for c in ov.columns if c not in rename]
assert not missing_names, f"overview columns without a clean name: {missing_names}"
trips = ov.rename(columns=rename)

# both date patterns seen in notebook 01: 2019-06-25_13-21-14 and 2019_12_08_09-38-26
trips["start_time"] = pd.to_datetime(trips.start_raw.astype(str).str.replace("_", "-"), format="%Y-%m-%d-%H-%M-%S")
assert trips.start_time.notna().all()
trips["series"] = trips.trip_id.str[4]
trips["season"] = trips.series.map({"A": "summer", "B": "winter"})
record("Inconsistent date formats", "both patterns parsed into start_time", "Overview.xlsx", len(trips))
record("Unlabelled column in Overview.xlsx", "named soc_used_frac (start SoC minus end SoC)", "Overview.xlsx",
       trips.soc_used_frac.notna().sum())
record("Duplicate trip start timestamps", "kept both; trip_id is the key, not start_time", "TripA30, TripA31", 0)

In [13]:
per_trip = clean.groupby("trip_id").agg(
    samples=("row_idx", "size"),
    duration_min=("time_s", lambda s: round(s.iloc[-1] / 60, 2)),
    soc_gap_samples=("soc_gap", "sum"),
    heating_extreme_samples=("heating_extreme", "sum"),
)
per_trip = per_trip.join(phase_minutes.add_suffix("_min"))
trips = trips.merge(per_trip, left_on="trip_id", right_index=True, how="left", validate="one_to_one")
trips = trips[["trip_id", "series", "season", "start_time", "route", "weather", "ambient_temp_start_c",
               "batt_temp_start_c", "batt_temp_end_c", "soc_start_frac", "soc_end_frac", "soc_used_frac",
               "target_cabin_temp_c", "fan", "distance_km", "duration_min_overview", "duration_min", "samples",
               "driving_min", "stopped_min", "charging_min", "soc_gap_samples", "heating_extreme_samples", "note"]]
trips.head()

,trip_id,series,season,start_time,route,weather,ambient_temp_start_c,batt_temp_start_c,batt_temp_end_c,soc_start_frac,soc_end_frac,soc_used_frac,target_cabin_temp_c,fan,distance_km,duration_min_overview,duration_min,samples,driving_min,stopped_min,charging_min,soc_gap_samples,heating_extreme_samples,note
0,TripA01,A,summer,2019-06-25 13:21:14,Munich East,sunny,25.5,21.0,22.0,0.863,0.803,0.060,23.0,"Automatic, Level 1",7.427690,16.820000,16.82,10090,14.7,2.1,0.0,0,0,NaN
1,TripA02,A,summer,2019-06-25 14:05:31,Munich East,sunny,32.0,23.0,26.0,0.803,0.673,0.130,23.0,"Automatic, Level 1",23.509709,23.550000,23.55,14130,23.6,0.0,0.0,0,0,Target Cabin Temperature changed
2,TripA03,A,summer,2019-06-28 10:02:15,Munich East,sunny,21.5,24.0,25.0,0.835,0.751,0.084,27.0,"Automatic, Level 1",12.820846,11.180000,11.18,6706,10.7,0.5,0.0,0,0,Target Cabin Temperature changed
3,TripA04,A,summer,2019-06-28 10:13:30,Munich East,sunny,24.0,25.0,27.0,0.751,0.667,0.084,22.0,"Automatic, Level 1",10.727491,6.870000,6.87,4124,6.3,0.6,0.0,0,0,NaN
4,TripA05,A,summer,2019-06-28 10:20:26,Munich East,sunny,24.5,27.0,27.0,0.667,0.602,0.065,24.0,"Automatic, Level 1",12.393223,22.776667,22.78,13667,21.3,1.5,0.0,0,0,NaN


## 6 · Load into DuckDB

| Schema | Table | Content |
|---|---|---|
| `raw` | `trip_signals` | every sample, clean column names, values exactly as recorded |
| `clean` | `trip_signals` | every sample after the fixes, with `phase`, `soc_filled`, `soc_gap`, `heating_extreme`, `heating_can_invalid`, `coolant_flow_lph` |
| `clean` | `trips` | one row per trip |
| `meta` | `column_map`, `cleaning_log`, `quality_checks` | how the data got here |

In [14]:
cleaning_log = pd.DataFrame(log)
if DB_PATH.exists():
    DB_PATH.unlink()  # rebuilt from the raw files on every run
con = duckdb.connect(str(DB_PATH))
for schema in ["raw", "clean", "meta"]:
    con.execute(f"CREATE SCHEMA {schema}")

for table, frame in {"raw.trip_signals": raw, "clean.trip_signals": clean, "clean.trips": trips,
                      "meta.column_map": column_map, "meta.cleaning_log": cleaning_log}.items():
    con.register("frame_view", frame)
    con.execute(f"CREATE TABLE {table} AS SELECT * FROM frame_view")
    con.unregister("frame_view")
con.sql("SELECT schema_name, table_name, estimated_size AS rows, column_count FROM duckdb_tables() ORDER BY 1, 2").df()

,schema_name,table_name,rows,column_count
0,clean,trip_signals,1094793,57
1,clean,trips,70,24
2,meta,cleaning_log,11,4
3,meta,column_map,49,2
4,raw,trip_signals,1094793,51


## 7 · Data-quality checks

Each check is a SQL query that must return **0**. The notebook stops if any check fails.

In [15]:
def q(sql):
    return con.execute(sql).fetchone()[0]


expected_rows = sum(1 for p in trip_files for _ in open(p, encoding="latin-1")) - len(trip_files)
checks = {
    "raw and clean have the same number of rows":
        "SELECT abs((SELECT count(*) FROM raw.trip_signals) - (SELECT count(*) FROM clean.trip_signals))",
    "no rows lost while reading the CSV files":
        f"SELECT abs((SELECT count(*) FROM raw.trip_signals) - {expected_rows})",
    "one row per trip and timestamp":
        "SELECT count(*) - count(DISTINCT (trip_id, time_s)) FROM clean.trip_signals",
    "time strictly increasing within each trip":
        "SELECT count(*) FROM (SELECT time_s - lag(time_s) OVER (PARTITION BY trip_id ORDER BY row_idx) AS dt "
        "FROM clean.trip_signals) WHERE dt <= 0",
    "every trip has metadata and every metadata row has samples":
        "SELECT (SELECT count(DISTINCT trip_id) FROM clean.trip_signals WHERE trip_id NOT IN "
        "(SELECT trip_id FROM clean.trips)) + (SELECT count(*) FROM clean.trips WHERE samples IS NULL)",
    "no values outside physical limits":
        "SELECT count(*) FROM clean.trip_signals WHERE " + " OR ".join(
            f"({c} IS NOT NULL AND {c} NOT BETWEEN {lo} AND {hi})" for c, (lo, hi) in LIMITS.items()),
    "phase is always one of driving / stopped / charging":
        "SELECT count(*) FROM clean.trip_signals WHERE phase NOT IN ('driving', 'stopped', 'charging') OR phase IS NULL",
    "SoC is missing only where flagged as a gap":
        "SELECT count(*) FROM clean.trip_signals WHERE soc_pct IS NULL AND NOT soc_gap",
    "core battery signals complete (voltage, current, temperature)":
        "SELECT count(*) FROM clean.trip_signals WHERE batt_voltage_v IS NULL OR batt_current_a IS NULL "
        "OR batt_temp_c IS NULL",
    "raw values unchanged outside logged fixes (voltage, current, speed)":
        "SELECT count(*) FROM raw.trip_signals r JOIN clean.trip_signals c USING (trip_id, row_idx) "
        "WHERE r.batt_voltage_v IS DISTINCT FROM c.batt_voltage_v OR r.batt_current_a IS DISTINCT FROM c.batt_current_a "
        "OR r.speed_kmh IS DISTINCT FROM c.speed_kmh",
}
results = pd.DataFrame([{"check": name, "failures": q(sql)} for name, sql in checks.items()])
results["passed"] = results.failures == 0
con.register("frame_view", results)
con.execute("CREATE TABLE meta.quality_checks AS SELECT * FROM frame_view")
con.unregister("frame_view")
print(results.to_string(index=False))
assert results.passed.all(), "a data-quality check failed"

                                                              check  failures  passed
                         raw and clean have the same number of rows         0    True
                           no rows lost while reading the CSV files         0    True
                                     one row per trip and timestamp         0    True
                          time strictly increasing within each trip         0    True
         every trip has metadata and every metadata row has samples         0    True
                                  no values outside physical limits         0    True
                phase is always one of driving / stopped / charging         0    True
                         SoC is missing only where flagged as a gap         0    True
      core battery signals complete (voltage, current, temperature)         0    True
raw values unchanged outside logged fixes (voltage, current, speed)         0    True


## 8 · Cleaning log

In [16]:
cleaning_log.to_csv(REPORTS / "02_cleaning_log.csv", index=False)
results.to_csv(REPORTS / "02_quality_checks.csv", index=False)

handled = set(cleaning_log.issue)
info_only = {"Current sign convention", "Two different SoC signals", "Coarse battery temperature"}
header_issues = {i for i in register.issue if i.startswith("Header:")}
open_issues = sorted(set(register.issue) - handled - info_only - header_issues - {"SoC jump between consecutive samples",
                                                                                     "Trip count differs from dataset description"})
print("issues from notebook 01 not handled here:", open_issues or "none")
cleaning_log

issues from notebook 01 not handled here: none


,issue,action,trips,values_changed
0,Value outside physical limits,"soc_pct outside [1, 100] set to missing",TripB14,1
1,Missing values in key signals,"SoC: left missing, flagged (gaps <= 5 s filled, longer flagged soc_gap)","TripB07, TripB09, TripB28, TripB34",30748
2,Missing values in key signals,"SoC: nearest value (trip edge) (gaps <= 5 s filled, longer flagged soc_gap)","TripB05, TripB06, TripB08, TripB10, TripB11, TripB12, TripB13, TripB14, TripB18, TripB...",46
3,Heating power spike,"kept: four independent signals agree, so it is a real event; flagged in heating_extrem...",TripB02,0
4,Column entirely empty,coolant flow never logged in these trips: left missing,"TripB05, TripB06",0
5,Coolant flow offset,added coolant_flow_lph = raw + 500 (raw column kept),winter trips,598759
6,Stale CAN heating power (found in notebook 04),"heating_power_can_kw > 0.5 kW while requested < 100 W set to missing, flagged heating_...","TripA03, TripA04, TripA05, TripA06, TripA08, TripA09, TripA13, TripA14, TripA18, TripA20",77442
7,Trips contain charging phases,"phase label added (charging = parked, > +20 A for >= 60 s)","TripB04, TripB09",39361
8,Inconsistent date formats,both patterns parsed into start_time,Overview.xlsx,70
9,Unlabelled column in Overview.xlsx,named soc_used_frac (start SoC minus end SoC),Overview.xlsx,70


In [17]:
con.close()
print(f"warehouse written: {DB_PATH.relative_to(ROOT)} ({DB_PATH.stat().st_size / 1e6:.0f} MB)")

warehouse written: data\warehouse\volvo.duckdb (91 MB)


## 9 · Summary

- **Schema:** 49 raw headers (summer and winter variants, typos and a duplicate) mapped to one clean schema of named, unit-suffixed columns.
  Winter-only signals stay empty for summer trips.
- **Fixes:** each one is in `meta.cleaning_log` and `reports/02_cleaning_log.csv`:
  - one out-of-range SoC value set to missing;
  - short SoC gaps of up to 5 s filled (mostly a single missing first or last sample), and the four long gaps (B07, B09, B28, B34) left missing and flagged;
  - the TripB02 heating event kept, because four independent signals confirm it, and flagged instead;
  - the coolant-flow offset converted;
  - stale CAN heating-power readings in summer trips (heating reported while none was requested) set to missing and flagged. This was found in the EDA, notebook 04.
- **Header issues** are solved by the column map. The **SoC jump** in TripB14 came from the same final-row zero, so it is fixed by 3.1.
  The **trip count** (70, not 72) needs no fix.
- **Phases:** every sample is labelled `driving`, `stopped` or `charging`, so charging no longer mixes into driving analysis.
- **Warehouse:** `data/warehouse/volvo.duckdb` (raw, clean and meta schemas). All data-quality checks pass, and the notebook stops if one fails.